# Brain–Behavior Analysis With an INR Residualization Parameter

Set `RESIDUALIZE_FOR_INR` in the parameter cell to run the matched analysis with or without income-to-needs ratio (INR) as an FC covariate. All other analysis steps are identical.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg
from matplotlib.patches import Patch
from sklearn.linear_model import LinearRegression
from sklearn.metrics import rand_score
from statsmodels.stats.multitest import multipletests

# Use a colorblind-friendly seaborn style for all figures in this notebook.
sns.set_palette('colorblind')
sns.set_style('whitegrid')

# Hide warnings from exploratory notebook output while keeping code execution unchanged.
warnings.filterwarnings('ignore')


In [ ]:
# Notebook-local data wrangling and statistics helpers.
INVALID_CODES = (555, 777, 888, 999)

STANDARD_FC_COVARIATES = [
    'demo_sex_v2',
    'interview_age',
    'site_id_l',
    'ehi1b',
    'mean_fd_0.20',
]

DEFAULT_CATEGORICAL_COVARIATES = {
    'demo_sex_v2',
    'site_id_l',
    'ehi1b'
}


def standardize_subject_id(subject_ids):
    return (
        subject_ids.astype(str)
        .str.replace('_', '', regex=False)
        .str.replace('^sub-', '', regex=True)
    )


def load_motion_qa(
    motion_qa_path,
    motion_column = 'mean_fd_0.20',
):
    motion_qa = pd.read_csv(motion_qa_path)

    motion_qa = motion_qa[['src_subject_id', motion_column]].copy()
    motion_qa['src_subject_id'] = standardize_subject_id(motion_qa['src_subject_id'])
    motion_qa = motion_qa.drop_duplicates(subset=['src_subject_id'])
    motion_qa[motion_column] = pd.to_numeric(motion_qa[motion_column], errors='coerce')
    return motion_qa


def merge_motion_qa(
    df,
    motion_qa_path,
    motion_column = 'mean_fd_0.20',
    how = 'left',
):
    motion_qa = load_motion_qa(motion_qa_path, motion_column=motion_column)

    merged = df.copy()
    merged['src_subject_id'] = standardize_subject_id(merged['src_subject_id'])

    if motion_column in merged.columns:
        existing_motion = merged.groupby('src_subject_id')[motion_column].first()
        merged = merged.drop(columns=[motion_column])
        merged = merged.merge(motion_qa, on='src_subject_id', how=how)
        merged[motion_column] = merged[motion_column].fillna(
            merged['src_subject_id'].map(existing_motion)
        )
    else:
        merged = merged.merge(motion_qa, on='src_subject_id', how=how)

    return merged


def _complete_covariate_mask(
    df,
    covariates,
    categorical_covariates = DEFAULT_CATEGORICAL_COVARIATES,
):
    categorical_covariates = set(categorical_covariates)
    complete = pd.Series(True, index=df.index)

    for covariate in covariates:
        is_categorical = df[covariate].dtype == 'object' or covariate in categorical_covariates
        if is_categorical:
            complete &= df[covariate].notna()
        else:
            complete &= pd.to_numeric(df[covariate], errors='coerce').notna()

    return complete

def encode_regression_covariates(
    df,
    covariates,
    categorical_covariates = DEFAULT_CATEGORICAL_COVARIATES,
):
    encoded_parts = []
    categorical_covariates = set(categorical_covariates)

    for covariate in covariates:
        if df[covariate].dtype == 'object' or covariate in categorical_covariates:
            encoded_parts.append(
                pd.get_dummies(
                    df[covariate],
                    prefix=covariate,
                    drop_first=True,
                    dtype=float,
                )
            )
        else:
            encoded_parts.append(
                pd.to_numeric(df[covariate], errors='coerce').to_frame(covariate)
            )

    if not encoded_parts:
        return pd.DataFrame(index=df.index)
    return pd.concat(encoded_parts, axis=1)


def residualize_fc_profiles(
    df,
    fc_columns,
    covariates = STANDARD_FC_COVARIATES,
):
    df = df.copy()
    covariate_complete = _complete_covariate_mask(df, covariates)

    validity_groups = {}
    for column in fc_columns:
        valid_idx = df[column].notnull() & covariate_complete
        key = valid_idx.to_numpy(dtype=np.bool_).tobytes()
        if key not in validity_groups:
            validity_groups[key] = (valid_idx, [])
        validity_groups[key][1].append(column)

    residual_frames = []
    for valid_idx, columns in validity_groups.values():
        output_columns = [column + '_resid' for column in columns]
        residuals = pd.DataFrame(np.nan, index=df.index, columns=output_columns)
        if valid_idx.sum() > 0:
            observed = df.loc[valid_idx, columns].to_numpy()
            covariate_matrix = encode_regression_covariates(
                df.loc[valid_idx],
                covariates,
            )
            if covariate_matrix.shape[1] == 0:
                predicted = np.tile(observed.mean(axis=0), (len(observed), 1))
            else:
                model = LinearRegression()
                model.fit(covariate_matrix, observed)
                predicted = model.predict(covariate_matrix)
            residuals.loc[valid_idx, output_columns] = observed - predicted
        residual_frames.append(residuals)

    if residual_frames:
        df = pd.concat([df, *residual_frames], axis=1)

    return df


def calculate_effect_size(
    data,
    group_col = 'kmeans_2_consensus',
    value_col = 'value',
):
    group_values = sorted(data[group_col].dropna().unique())
    if len(group_values) != 2:
        return np.nan
    group_1 = data[data[group_col] == group_values[0]][value_col].dropna()
    group_2 = data[data[group_col] == group_values[1]][value_col].dropna()

    n1 = len(group_1)
    n2 = len(group_2)
    if n1 < 2 or n2 < 2:
        return np.nan

    std_1 = group_1.std(ddof=1)
    std_2 = group_2.std(ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * std_1**2 + (n2 - 1) * std_2**2) / (n1 + n2 - 2))
    if pooled_std == 0 or pd.isna(pooled_std):
        return np.nan

    return (group_1.mean() - group_2.mean()) / pooled_std


def clean_group_measure(
    df,
    measure,
    group_col = 'kmeans_2_consensus',
    invalid_codes = INVALID_CODES,
):
    return (
        df[[group_col, measure]]
        .replace(list(invalid_codes), np.nan)
        .dropna()
        .copy()
    )


def compute_correlations(
    df,
    measures,
    fc_columns,
    required_nonmissing=None,
):
    results = []
    required_nonmissing = list(required_nonmissing or [])

    for measure in measures:
        for column in fc_columns:
            analysis_columns = list(dict.fromkeys([column, measure, *required_nonmissing]))
            temp_df = df[analysis_columns].dropna()

            if (
                len(temp_df) < 2
                or temp_df[column].nunique(dropna=True) < 2
                or temp_df[measure].nunique(dropna=True) < 2
            ):
                r_value = np.nan
                p_value = np.nan
            else:
                r_value, p_value = stats.pearsonr(temp_df[column], temp_df[measure])

            results.append({
                'measure': measure,
                'col': column,
                'r': r_value,
                'p': p_value,
                'n': len(temp_df),
            })

    return pd.DataFrame(results)


def apply_multiple_comparison_corrections(
    results_df,
    alpha = 0.05,
    fdr_group_col=None,
):
    if results_df.empty:
        return results_df.assign(
            p_bonf=pd.Series(dtype=float),
            sig_bonf=pd.Series(dtype=bool),
            p_fdr=pd.Series(dtype=float),
            sig_fdr=pd.Series(dtype=bool),
        )

    corrected = results_df.copy()
    n_tests = len(corrected)
    corrected['p_bonf'] = np.minimum(corrected['p'] * n_tests, 1.0)
    corrected['sig_bonf'] = corrected['p_bonf'] < alpha

    valid_p = corrected['p'].notna()
    corrected['p_fdr'] = np.nan
    corrected['sig_fdr'] = False

    if fdr_group_col is None:
        fdr_families = pd.Series('all', index=corrected.index)
    else:
        fdr_families = corrected[fdr_group_col].astype('string').fillna('<missing>')

    for family in fdr_families.unique():
        family_valid_p = valid_p & fdr_families.eq(family)
        if family_valid_p.any():
            reject, p_fdr, _, _ = multipletests(
                corrected.loc[family_valid_p, 'p'],
                alpha=alpha,
                method='fdr_bh',
            )
            corrected.loc[family_valid_p, 'p_fdr'] = p_fdr
            corrected.loc[family_valid_p, 'sig_fdr'] = reject

    return corrected


def pairwise_rand_scores(df, columns):
    rand_values = []
    for i, column_i in enumerate(columns):
        for column_j in columns[i + 1:]:
            rand_values.append(rand_score(df[column_i], df[column_j]))
    return rand_values


# Data Wrangling

In [ ]:
# Choose whether this notebook controls for income-to-needs ratio (INR) in brain-behavior tests.
RESIDUALIZE_FOR_INR = False
ANALYSIS_LABEL = 'with INR covariate' if RESIDUALIZE_FOR_INR else 'without INR covariate'
INR_VARIABLE = 'inr'
INR_RESIDUALIZATION_COVARIATE = 'inr'

# Load the clustered FC/behavior table and merge motion QA values without dropping subjects lacking QA rows.
DATA_PATH = (
    '/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/midb61_meanFC_clusters_motion_inr_resid_2026-07-14_15-40.csv'
    if RESIDUALIZE_FOR_INR
    else '/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/midb61_meanFC_clusters_motion_resid_2026-07-14_17-56.csv'
)
WRANGLED_DATA_PATH = '/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/wrangled_pMTG_FC_data_midb61_meanFC.csv'
MOTION_QA_PATH = '/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/motion_QA_results.csv'
PCA_RESULTS_DIR = Path('/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/pca_results')
EFA_SCORE_MODEL_NAME = 'with_ses_residualization' if RESIDUALIZE_FOR_INR else 'without_ses_residualization'
EFA_SCORE_PREFIX = 'ses_cognitive' if RESIDUALIZE_FOR_INR else 'no_ses_cognitive'
EFA_SCORE_PATH = PCA_RESULTS_DIR / f'efa_cognitive_scores_{EFA_SCORE_MODEL_NAME}.csv'

df = pd.read_csv(DATA_PATH)
df = merge_motion_qa(df, MOTION_QA_PATH, how='left')


def merge_optional_subject_columns(df, source_path, columns, source_label):
    # Bring in descriptive variables needed for subtype checks without making them model covariates.
    available_columns = pd.read_csv(source_path, nrows=0).columns
    columns_to_merge = [column for column in columns if column in available_columns]
    if not columns_to_merge:
        print(f'No optional {source_label} columns available to merge: {columns}')
        return df

    optional_data = pd.read_csv(
        source_path,
        usecols=lambda column: column == 'src_subject_id' or column in columns_to_merge,
    )
    optional_data['src_subject_id'] = standardize_subject_id(optional_data['src_subject_id'])
    optional_data = optional_data.drop_duplicates(subset='src_subject_id')

    overlapping_columns = [column for column in columns_to_merge if column in df.columns]
    df = df.drop(columns=overlapping_columns, errors='ignore')
    df['src_subject_id'] = standardize_subject_id(df['src_subject_id'])
    return df.merge(optional_data, on='src_subject_id', how='left', validate='many_to_one')


# Keep only Fisher-z FC variables used in analyses; generated _full networks stay in source files only.
full_fc_cols = [column for column in df.columns if '_fz' in str(column) and '_full' in str(column)]
if full_fc_cols:
    df = df.drop(columns=full_fc_cols)
print(f'Dropped {len(full_fc_cols)} generated full-network Fisher-z FC columns from the analysis dataframe.')

# Add non-covariate subtype comparison variables requested in the manuscript.
df = merge_optional_subject_columns(df, MOTION_QA_PATH, ['good_frames_0.20'], 'motion QA')
df = merge_optional_subject_columns(df, WRANGLED_DATA_PATH, ['accult_q2_y'], 'wrangled data')
efa_scores_df = pd.read_csv(EFA_SCORE_PATH)
efa_scores_df['src_subject_id'] = standardize_subject_id(efa_scores_df['src_subject_id'])
efa_scores_df = efa_scores_df.drop_duplicates(subset='src_subject_id')
cognitive_efa_measures = [
    col for col in efa_scores_df.columns if col.startswith(f'{EFA_SCORE_PREFIX}_EF')
]


df['src_subject_id'] = standardize_subject_id(df['src_subject_id'])
df = df.drop(columns=[col for col in cognitive_efa_measures if col in df.columns])
df = df.merge(
    efa_scores_df[['src_subject_id', *cognitive_efa_measures]],
    on='src_subject_id',
    how='left',
)
missing_efa_rows = df[cognitive_efa_measures].isna().all(axis=1).sum()

# Report columns and missing motion values so input-file mismatches are visible before analysis.
missing_motion = df['mean_fd_0.20'].isna().sum() if 'mean_fd_0.20' in df.columns else len(df)
print(f'Running brain-behavior analysis {ANALYSIS_LABEL}')
print(f'Missing mean_fd_0.20 values after motion QA merge: {missing_motion}')
for col in df.columns:
    print(col)


# Calculate INR for SES Analyses

In [ ]:

# Confirm INR columns only when this notebook actually residualizes for INR.
required_inr_cols = [INR_VARIABLE, 'inr_missing', 'poverty_line_2017'] if RESIDUALIZE_FOR_INR else []
inr_display_cols = [
    'demo_comb_income_v2',
    'demo_roster_v2',
    'poverty_line_2017',
    INR_VARIABLE,
    'inr_missing',
]
if RESIDUALIZE_FOR_INR and INR_RESIDUALIZATION_COVARIATE not in inr_display_cols:
    inr_display_cols.append(INR_RESIDUALIZATION_COVARIATE)

if RESIDUALIZE_FOR_INR:
    missing_inr_cols = [col for col in required_inr_cols if col not in df.columns]
    if missing_inr_cols:
        wrangled_inr_columns = ['src_subject_id'] + list(dict.fromkeys(inr_display_cols))
        wrangled_inr = pd.read_csv(
            WRANGLED_DATA_PATH,
            usecols=lambda column: column in wrangled_inr_columns,
        )
        wrangled_inr['src_subject_id'] = standardize_subject_id(wrangled_inr['src_subject_id'])
        wrangled_inr = wrangled_inr.drop_duplicates(subset='src_subject_id')

        existing_inr_cols = [col for col in wrangled_inr.columns if col != 'src_subject_id' and col in df.columns]
        df = df.drop(columns=existing_inr_cols)
        df['src_subject_id'] = standardize_subject_id(df['src_subject_id'])
        df = df.merge(wrangled_inr, on='src_subject_id', how='left', validate='many_to_one')

    missing_inr_cols = [col for col in required_inr_cols if col not in df.columns]

    print(df[[col for col in inr_display_cols if col in df.columns]].head())
    print(f'Missing raw INR values excluded from INR-residualized analyses: {df[INR_VARIABLE].isna().sum()}')
    print(f'Observed INR covariate mean: {df[INR_RESIDUALIZATION_COVARIATE].mean():.4f}')
else:
    present_inr_cols = [col for col in inr_display_cols if col in df.columns]
    if present_inr_cols:
        print(df[present_inr_cols].head())
    print('INR is not required for this no-INR-residualization brain-behavior analysis.')


# Scanner Software Version Checks

In [ ]:

# Test whether age, sex, and available SES variables differ across scanner software versions.
software_col = 'mri_info_softwareversion'
scanner_check_vars = ['interview_age', 'demo_sex_v2']
if INR_VARIABLE in df.columns:
    scanner_check_vars.append(INR_VARIABLE)
scanner_df = df[[software_col] + scanner_check_vars].copy()
scanner_df = scanner_df.replace(list(INVALID_CODES), np.nan)

print('Scanner software version counts:')
print(scanner_df[software_col].value_counts(dropna=False).sort_index())

software_test_results = []


def record_software_test(measure, test_name, statistic, p_value):
    # Store uncorrected p-values for scanner-software diagnostics.
    software_test_results.append({
        'measure': measure,
        'test': test_name,
        'statistic': statistic,
        'p': p_value,
    })


def test_continuous_by_software(data, measure):
    # Summarize the continuous measure in each software-version group.
    summary = data.groupby(software_col)[measure].agg(['count', 'mean', 'std']).sort_index()
    print()
    print(f'{measure} by scanner software version:')
    print(summary)

    grouped_values = [
        group[measure].dropna()
        for _, group in data[[software_col, measure]].dropna().groupby(software_col)
    ]
    valid_groups = [values for values in grouped_values if len(values) >= 2]
    if len(valid_groups) < 2:
        print(f'Not enough software-version groups to test {measure}.')
        return

    f_stat, anova_p = stats.f_oneway(*valid_groups)
    h_stat, kruskal_p = stats.kruskal(*valid_groups)
    print(f'ANOVA for {measure}: F = {f_stat:.4f}, p = {anova_p:.6g}')
    print(f'Kruskal-Wallis for {measure}: H = {h_stat:.4f}, p = {kruskal_p:.6g}')
    record_software_test(measure, 'ANOVA', f_stat, anova_p)
    record_software_test(measure, 'Kruskal-Wallis', h_stat, kruskal_p)


def test_categorical_by_software(data, measure):
    # Cross-tabulate categorical measures by software version and test independence.
    contingency = pd.crosstab(data[software_col], data[measure])
    print()
    print(f'{measure} by scanner software version:')
    print(contingency)

    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        print(f'Not enough category variation to test {measure}.')
        return

    chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
    denom = contingency.to_numpy().sum() * (min(contingency.shape) - 1)
    cramer_v = np.sqrt(chi2 / denom) if denom > 0 else np.nan
    print(f'Chi-square for {measure}: chi2 = {chi2:.4f}, dof = {dof}, p = {p_value:.6g}')
    print(f"Cramer's V for {measure}: {cramer_v:.4f}")
    record_software_test(measure, 'Chi-square', chi2, p_value)


test_continuous_by_software(scanner_df, 'interview_age')
test_categorical_by_software(scanner_df, 'demo_sex_v2')
if INR_VARIABLE in scanner_df.columns:
    test_continuous_by_software(scanner_df, INR_VARIABLE)

software_test_results_df = pd.DataFrame(software_test_results)
if not software_test_results_df.empty:
    print()
    print('Scanner software version test summary (uncorrected non-task diagnostics):')
    print(software_test_results_df)


# FC Residualization

In [ ]:
# Recreate FC residuals inside this notebook from the raw pMTG-network FC columns.
raw_fc_cols = [col for col in df.columns if col.endswith('_fz') and '_full' not in col]

# FC residuals remove demographics, site, motion QA mean FD, and optionally INR in one model.
FC_COVARIATES = [
    'demo_sex_v2',
    'interview_age',
    'site_id_l',
    'ehi1b',
    'mean_fd_0.20',
]
if RESIDUALIZE_FOR_INR:
    FC_COVARIATES = FC_COVARIATES + [INR_RESIDUALIZATION_COVARIATE]

missing_fc_covariates = [covariate for covariate in FC_COVARIATES if covariate not in df.columns]

# Drop any residual columns already present in the CSV so these are created fresh here.
existing_fc_resid_cols = [f'{col}_resid' for col in raw_fc_cols if f'{col}_resid' in df.columns]
if existing_fc_resid_cols:
    df = df.drop(columns=existing_fc_resid_cols)

print('Residualizing FC columns for:', FC_COVARIATES)
df = residualize_fc_profiles(df, raw_fc_cols, covariates=FC_COVARIATES)
analysis_fc_cols = [f'{col}_resid' for col in raw_fc_cols]

print(f'Raw FC columns residualized: {len(raw_fc_cols)}')
print(f'Number of FC columns used in brain-behavior tests: {len(analysis_fc_cols)}')


# Comparing groups on EF measures

In [ ]:
# Recreate the individual residualized task scores used as EFA inputs in PCA_tasks.
COGNITIVE_TASK_MEASURES = [
    'ravlt_immediate_2y',
    'ravlt_short_delay_2y',
    'ravlt_long_delay_2y',
    'nihtbx_picvocab_uncorrected_2y',
    'nihtbx_reading_uncorrected_2y',
    'nihtbx_picture_uncorrected_2y',
]
COGNITIVE_TASK_LABELS = {
    'ravlt_immediate_2y': 'RAVLT Immediate Recall',
    'ravlt_short_delay_2y': 'RAVLT Short Delayed Recall',
    'ravlt_long_delay_2y': 'RAVLT Long Delayed Recall',
    'nihtbx_picvocab_uncorrected_2y': 'Picture Vocabulary',
    'nihtbx_reading_uncorrected_2y': 'Oral Reading Recognition',
    'nihtbx_picture_uncorrected_2y': 'Picture Sequence Memory',
}
COGNITIVE_TASK_RESIDUAL_SUFFIX = 'ses' if RESIDUALIZE_FOR_INR else 'no_ses'
COGNITIVE_TASK_RESIDUAL_COVARIATES = ['interview_age', 'demo_sex_v2', 'site_id_l', 'ehi1b']
if RESIDUALIZE_FOR_INR:
    COGNITIVE_TASK_RESIDUAL_COVARIATES = COGNITIVE_TASK_RESIDUAL_COVARIATES + [INR_RESIDUALIZATION_COVARIATE]


def residualize_one_cognitive_task(df, measure, covariates, output_col):
    # Residualize a raw task score with the same covariates used before EFA.
    residualized = df.copy()
    if output_col in residualized.columns:
        residualized = residualized.drop(columns=[output_col])

    task_values = pd.to_numeric(
        residualized[measure].replace(list(INVALID_CODES), np.nan),
        errors='coerce',
    )
    complete = task_values.notna()
    for covariate in covariates:
        is_categorical = residualized[covariate].dtype == 'object' or covariate in DEFAULT_CATEGORICAL_COVARIATES
        if is_categorical:
            complete &= residualized[covariate].notna()
        else:
            complete &= pd.to_numeric(residualized[covariate], errors='coerce').notna()

    residualized[output_col] = np.nan
    if complete.sum() == 0:
        return residualized

    observed = task_values.loc[complete].to_numpy()
    covariate_matrix = encode_regression_covariates(residualized.loc[complete], covariates)
    if covariate_matrix.shape[1] == 0:
        predicted = np.repeat(observed.mean(), len(observed))
    else:
        model = LinearRegression()
        model.fit(covariate_matrix, observed)
        predicted = model.predict(covariate_matrix)

    residualized.loc[complete, output_col] = observed - predicted
    return residualized


def residualize_cognitive_task_scores(df, measures, covariates, output_suffix):
    residualized = df.copy()
    output_cols = []
    for measure in measures:
        output_col = f'{measure}_{output_suffix}_resid'
        residualized = residualize_one_cognitive_task(
            residualized,
            measure,
            covariates,
            output_col,
        )
        output_cols.append(output_col)
    return residualized, output_cols


df, cognitive_task_resid_cols = residualize_cognitive_task_scores(
    df,
    COGNITIVE_TASK_MEASURES,
    COGNITIVE_TASK_RESIDUAL_COVARIATES,
    COGNITIVE_TASK_RESIDUAL_SUFFIX,
)
cognitive_task_labels = {
    f'{measure}_{COGNITIVE_TASK_RESIDUAL_SUFFIX}_resid': label
    for measure, label in COGNITIVE_TASK_LABELS.items()
}

print('Individual cognitive task residualization covariates:', COGNITIVE_TASK_RESIDUAL_COVARIATES)
print('Residualized cognitive task scores used to construct EFA variables:', cognitive_task_resid_cols)
for measure in cognitive_task_resid_cols:
    print(f'Sample size for {measure}: {df[measure].notna().sum()}')

# Use matched exploratory factor scores as cognitive task summaries.
for measure in cognitive_efa_measures:
    df[measure] = pd.to_numeric(df[measure], errors='coerce')

cognitive_group_measures = cognitive_efa_measures.copy()
analysis_measures = cognitive_efa_measures.copy()

print('Matched cognitive EFA measures:', cognitive_efa_measures)
print('Brain-behavior cognitive EFA measures:', analysis_measures)


In [ ]:

# Compare latent retrieval factors, individual EFA-input task scores, demographics, and motion/language variables across KMeans subtype groups.
CLUSTER_GROUP_COL = 'kmeans_2_consensus'
print(f'Cluster group column for subtype comparisons: {CLUSTER_GROUP_COL}')
cat_measures = ['tanner_stage', 'site_id_l', 'ehi1b', 'accult_q2_y']
cont_measures = ['good_frames_0.20', 'mean_fd_0.20']
if INR_VARIABLE in df.columns:
    cont_measures.append(INR_VARIABLE)


def calculate_ttest_result(df_subset, measure):
    # Welch's t-test is used for continuous subtype comparisons.
    group_values = sorted(df_subset[CLUSTER_GROUP_COL].dropna().unique())
    if len(group_values) != 2:
        print(f'Expected two {CLUSTER_GROUP_COL} groups to test {measure}, found {group_values}')
        return None
    cluster_1 = df_subset[df_subset[CLUSTER_GROUP_COL] == group_values[0]][measure]
    cluster_2 = df_subset[df_subset[CLUSTER_GROUP_COL] == group_values[1]][measure]
    if len(cluster_1) < 2 or len(cluster_2) < 2:
        print(f'Not enough data to test {measure}: n0={len(cluster_1)}, n1={len(cluster_2)}')
        return None

    t_stat, p_val = stats.ttest_ind(cluster_1, cluster_2, equal_var=False)
    cohen_d = calculate_effect_size(df_subset, group_col=CLUSTER_GROUP_COL, value_col=measure)
    return {
        'measure': measure,
        'test': 'Welch t-test',
        't_stat': t_stat,
        'p': p_val,
        'cohen_d': cohen_d,
        'group0': group_values[0],
        'group1': group_values[1],
        'n0': len(cluster_1),
        'n1': len(cluster_2),
    }


def print_ttest_result(result, p_bonf=None):
    if result is None:
        return
    measure_label = result['measure']
    if 'label' in result and pd.notna(result['label']):
        measure_label = result['label']
    p_eval = p_bonf if p_bonf is not None else result['p']
    significance = 'Significant' if p_eval < 0.05 else 'No significant'
    corrected_text = f", p_bonf = {p_bonf:.7f}" if p_bonf is not None else ''
    print(
        f"{significance} difference in {measure_label} between clusters: "
        f"t_stat = {result['t_stat']:.3f}, p-value = {result['p']:.7f}"
        f"{corrected_text}"
    )
    print(f"Cohen's d = {result['cohen_d']:.3f}")
    print()


def run_chi2(df_subset, measure):
    # Chi-square tests are used for categorical subtype comparisons.
    contingency = pd.crosstab(df_subset[CLUSTER_GROUP_COL], df_subset[measure])
    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        print(f'Not enough category variation to test {measure}')
        return

    chi2, p_val, dof, expected = stats.chi2_contingency(contingency)
    denom = df_subset.shape[0] * (min(contingency.shape) - 1)
    cramer_v = np.sqrt(chi2 / denom) if denom > 0 else np.nan

    significance = 'Significant' if p_val < 0.05 else 'No significant'
    print(
        f'{significance} difference in {measure} between clusters: '
        f'chi2 = {chi2:.3f}, p-value = {p_val:.7f}'
    )
    print(f"Cramer's V = {cramer_v:.3f}")
    print()


def clean_group_measure_if_available(df, measure):
    # Missing optional subtype variables are reported rather than silently skipped.
    if measure not in df.columns:
        print(f'Skipping {measure}: column is not available in this analysis dataframe.')
        return None
    return clean_group_measure(df, measure, group_col=CLUSTER_GROUP_COL)


cognitive_cluster_tests = []
for measure in cognitive_group_measures:
    temp_df = clean_group_measure_if_available(df, measure)
    if temp_df is None:
        continue
    result = calculate_ttest_result(temp_df, measure)
    if result is not None:
        cognitive_cluster_tests.append(result)

if cognitive_cluster_tests:
    cognitive_cluster_tests_df = apply_multiple_comparison_corrections(
        pd.DataFrame(cognitive_cluster_tests),
        alpha=0.05,
    )
    for _, result in cognitive_cluster_tests_df.iterrows():
        print_ttest_result(result, p_bonf=result['p_bonf'])
else:
    cognitive_cluster_tests_df = pd.DataFrame()

cognitive_task_cluster_tests = []
for measure in cognitive_task_resid_cols:
    temp_df = clean_group_measure_if_available(df, measure)
    if temp_df is None:
        continue
    result = calculate_ttest_result(temp_df, measure)
    if result is not None:
        result['label'] = cognitive_task_labels.get(measure, measure)
        cognitive_task_cluster_tests.append(result)

if cognitive_task_cluster_tests:
    cognitive_task_cluster_tests_df = apply_multiple_comparison_corrections(
        pd.DataFrame(cognitive_task_cluster_tests),
        alpha=0.05,
    )
    print('Individual residualized cognitive task score cluster tests, Bonferroni-corrected within task family:')
    for _, result in cognitive_task_cluster_tests_df.iterrows():
        print_ttest_result(result, p_bonf=result['p_bonf'])
    display(
        cognitive_task_cluster_tests_df[
            ['measure', 'label', 'test', 't_stat', 'p', 'p_bonf', 'sig_bonf', 'cohen_d', 'n0', 'n1']
        ]
    )
else:
    cognitive_task_cluster_tests_df = pd.DataFrame()

for measure in cat_measures:
    temp_df = clean_group_measure_if_available(df, measure)
    if temp_df is not None:
        run_chi2(temp_df, measure)

for measure in cont_measures:
    temp_df = clean_group_measure_if_available(df, measure)
    if temp_df is not None:
        result = calculate_ttest_result(temp_df, measure)
        print_ttest_result(result)


# Sample Sizes


In [ ]:
# Report the available sample for each cognitive factor and for INR.
for measure in analysis_measures:
    print(f'Sample size for {measure}: {df[measure].notna().sum()}')
print(f"Sample size for INR: {df[INR_VARIABLE].notna().sum()}")


# Network FC - EFA Significant Correlations

In [ ]:

# Report the cognitive EFA-FC test family size; FDR is applied after p-values are computed.
sample_columns = list(dict.fromkeys([*cognitive_efa_measures, *analysis_fc_cols]))
if RESIDUALIZE_FOR_INR:
    sample_columns.append(INR_VARIABLE)
n = df[sample_columns].dropna().shape[0]
n_cognitive_tests = len(cognitive_efa_measures) * len(analysis_fc_cols)
alpha = 0.05
trend_alpha = 0.001

dfree = n - 2
if dfree > 0:
    trend_t_crit = stats.t.ppf(1 - trend_alpha / 2, dfree)
    r_trend = np.sqrt(trend_t_crit**2 / (trend_t_crit**2 + dfree))
else:
    r_trend = np.nan

print('Sample size for complete cognitive EFA-FC family:', n)
print('Cognitive EFA-FC tests:', n_cognitive_tests)
print('FDR alpha for cognitive EFA-FC tests:', alpha)
print('Trend alpha:', trend_alpha)
print('Trend critical r:', r_trend)


In [ ]:
# Compute FC-by-cognitive-EFA correlations and FDR-correct the cognitive family only.
results_df = compute_correlations(
    df,
    cognitive_efa_measures,
    analysis_fc_cols,
    required_nonmissing=[INR_VARIABLE] if RESIDUALIZE_FOR_INR else None,
)
results_df = apply_multiple_comparison_corrections(results_df, alpha=0.05)
results_df['abs_r'] = results_df['r'].abs()
cognitive_results_df = results_df.copy()
cognitive_efa_results_df = cognitive_results_df.copy()

sig_cols = []
print()
print('FDR-significant cognitive EFA factor-FC correlations:')
fdr_results = cognitive_efa_results_df[cognitive_efa_results_df['sig_fdr']].sort_values('p_fdr')

if fdr_results.empty:
    print('None')
else:
    for _, row in fdr_results.iterrows():
        print(
            f"{row['col']} vs {row['measure']}: "
            f"r = {row['r']:.4f}, p = {row['p']:.4g}, p_fdr = {row['p_fdr']:.4g}, n = {row['n']}"
        )
        sig_cols.append(row['col'])

print()
print('Top 10 cognitive EFA factor-FC correlations by absolute r-value:')
top_efa_correlations = cognitive_efa_results_df.reindex(
    cognitive_efa_results_df['abs_r'].sort_values(ascending=False).index
).head(10)
display(top_efa_correlations)

print('Sample size range:', results_df['n'].min(), '-', results_df['n'].max())


In [ ]:
# Plot only FDR-significant cognitive EFA factor correlations.
plot_df = cognitive_efa_results_df[cognitive_efa_results_df['sig_fdr']].copy()

if plot_df.empty:
    print('No FDR-significant cognitive EFA factor correlations to plot.')
else:
    plot_df['network'] = plot_df['col'].str.split('_').str[0]
    plot_df['network'] = plot_df['network'].replace({'CO': 'AMN', 'Aud': 'AUD', 'Sal': 'SAL'})

    seed_net_raw = plot_df['col'].str.split('_').str[0]
    seed_hemi = plot_df['col'].str.split('_').str[1].str[0].str.upper()
    seed_lr = plot_df['col'].str.split('_').str[2]

    seed_net = seed_net_raw.replace({'CO': 'AMN', 'Aud': 'AUD', 'Sal': 'SAL'})
    seed_net = np.where((seed_net_raw == 'VAN') & (seed_hemi == 'L'), 'LANG', seed_net)

    plot_df['network'] = seed_net
    plot_df['hemi'] = seed_hemi
    plot_df['seed_label'] = np.where(
        seed_net_raw == 'VAN',
        seed_net + ' - ' + seed_lr + ' pMTG FC',
        seed_hemi + ' ' + seed_net + ' - ' + seed_lr + ' pMTG FC',
    )

    measure_map = {
        measure: f'EFA Factor {idx + 1}'
        for idx, measure in enumerate(cognitive_efa_measures)
    }
    plot_df['measure_label'] = plot_df['measure'].map(measure_map).fillna(plot_df['measure'])

    network_colors = {
        'DMN': '#fb2e2e', 'SMd': '#40cce9', 'LANG': '#2DB6B6FF', 'TPOLE': '#2e87ae',
        'AUD': '#ce8fff', 'AMN': '#8a2edb', 'DAN': '#2efe2e', 'FP': '#cdcd00ff',
        'MTL': '#89fd89', 'PMN': '#2e2eff', 'PON': '#a29c9cff', 'SAL': '#2e2e2e',
        'VAN': '#2DB6B6FF', 'SMl': '#ffa12e', 'VIS': '#2e2eb3',
    }

    heat = plot_df.pivot_table(index='seed_label', columns='measure_label', values='r', aggfunc='first')
    row_info = plot_df.drop_duplicates('seed_label').set_index('seed_label').loc[heat.index]

    blue_network_order = ['SMd', 'SMl', 'AUD', 'AMN', 'DAN', 'VAN', 'LANG']
    red_network_order = ['DMN', 'FP', 'MTL', 'PMN', 'PON', 'SAL']
    network_order = blue_network_order + red_network_order

    row_info['network_order'] = row_info['network'].map({network: i for i, network in enumerate(network_order)})
    heat = heat.loc[row_info.sort_values(['network_order', 'hemi']).index]
    row_info = row_info.loc[heat.index]
    row_networks = row_info['network']

    fig, ax = plt.subplots(figsize=(12, max(6, 0.3 * len(heat))))
    im = ax.imshow(heat, aspect='auto', cmap='coolwarm', vmin=-0.15, vmax=0.15)
    ax.grid(False)

    ax.set_xticks(range(len(heat.columns)))
    ax.set_xticklabels(heat.columns, rotation=45, ha='right', fontsize=11)
    ax.set_yticks(range(len(heat.index)))
    ax.set_yticklabels(heat.index, fontsize=11)

    for y, label in enumerate(ax.get_yticklabels()):
        net = row_networks.iloc[y]
        label.set_color(network_colors.get(net, 'black'))
        label.set_fontweight('bold')

    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            val = heat.iloc[i, j]
            if pd.notna(val):
                ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=8, fontweight='bold')

    legend_handles = []
    for network in network_order:
        if network in set(row_networks):
            if network == 'VAN':
                label = 'LANG/VAN'
            elif network == 'LANG':
                continue
            else:
                label = network
            legend_handles.append(Patch(color=network_colors[network], label=label))

    ax.legend(handles=legend_handles, title='Network', bbox_to_anchor=(1.25, 1), loc='upper left')
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('Pearson r', fontsize=12)

    ax.set_title(f'Cognitive EFA Factor-FC Correlations (FDR Corrected, {ANALYSIS_LABEL})', fontsize=18)
    ax.set_xlabel('Cognitive EFA Factor', fontsize=16)
    ax.set_ylabel('pMTG - Network Functional Connectivity Measure', fontsize=16)

    plt.tight_layout()
    plt.show()


In [ ]:
# Compare lateralized pMTG FC links with selected networks.
# This deliberately constructs only lateralized PMN/PON/DAN/MTL column names.
selected_fc_targets = [
    ('PMN', 'left', 'Left parietal memory network'),
    ('PMN', 'right', 'Right parietal memory network'),
    ('PON', 'left', 'Left parieto-occipital network'),
    ('PON', 'right', 'Right parieto-occipital network'),
    ('DAN', 'left', 'Left dorsal attention network'),
    ('DAN', 'right', 'Right dorsal attention network'),
    ('MTL', 'left', 'Left medial temporal network'),
]
selected_pmtg_seeds = [
    ('L', 'Left pMTG'),
    ('R', 'Right pMTG'),
]

selected_rows = []
missing_selected_cols = []
for network, target_hemi, target_label in selected_fc_targets:
    for pmtg_hemi, pmtg_label in selected_pmtg_seeds:
        fc_col = f'{network}_{target_hemi}_{pmtg_hemi}_fz_resid'
        fc_results = cognitive_efa_results_df[cognitive_efa_results_df['col'].eq(fc_col)].copy()
        if fc_results.empty:
            missing_selected_cols.append(fc_col)
            continue

        fc_results['target_network'] = target_label
        fc_results['pmtg_seed'] = pmtg_label
        fc_results['network'] = network
        fc_results['target_hemi'] = target_hemi.title()
        selected_rows.append(fc_results)

if missing_selected_cols:
    print('Missing selected lateralized FC columns:')
    for missing_col in missing_selected_cols:
        print(f'  {missing_col}')

if not selected_rows:
    print('No selected lateralized PMN/PON/DAN/MTL FC results found to plot.')
else:
    selected_fc_factor_df = pd.concat(selected_rows, ignore_index=True)

    first_efa_measure = cognitive_efa_measures[0]
    selected_fc_factor_df = selected_fc_factor_df[
        selected_fc_factor_df['measure'].eq(first_efa_measure)
    ].copy()

    measure_map = {first_efa_measure: 'EFA Factor 1'}
    target_order = [target_label for _, _, target_label in selected_fc_targets]
    pmtg_order = [pmtg_label for _, pmtg_label in selected_pmtg_seeds]
    factor_order = ['EFA Factor 1']

    selected_fc_factor_df['factor_label'] = selected_fc_factor_df['measure'].map(measure_map).fillna(selected_fc_factor_df['measure'])
    selected_fc_factor_df['relationship_strength'] = selected_fc_factor_df['r'].abs()
    selected_fc_factor_df['target_network'] = pd.Categorical(
        selected_fc_factor_df['target_network'],
        categories=target_order,
        ordered=True,
    )
    selected_fc_factor_df['pmtg_seed'] = pd.Categorical(
        selected_fc_factor_df['pmtg_seed'],
        categories=pmtg_order,
        ordered=True,
    )
    selected_fc_factor_df['factor_label'] = pd.Categorical(
        selected_fc_factor_df['factor_label'],
        categories=factor_order,
        ordered=True,
    )
    selected_fc_factor_df = selected_fc_factor_df.sort_values(['factor_label', 'target_network', 'pmtg_seed'])

    selected_pmtg_comparison_rows = []
    for factor_label in factor_order:
        for target_label in target_order:
            comparison = selected_fc_factor_df[
                selected_fc_factor_df['factor_label'].eq(factor_label)
                & selected_fc_factor_df['target_network'].eq(target_label)
            ].set_index('pmtg_seed')
            if not set(pmtg_order).issubset(comparison.index):
                continue

            left_strength = comparison.loc['Left pMTG', 'relationship_strength']
            right_strength = comparison.loc['Right pMTG', 'relationship_strength']
            stronger_seed = 'Left pMTG' if left_strength > right_strength else 'Right pMTG'
            if np.isclose(left_strength, right_strength, equal_nan=False):
                stronger_seed = 'Tie'

            selected_pmtg_comparison_rows.append({
                'factor_label': factor_label,
                'target_network': target_label,
                'left_pmtg_abs_r': left_strength,
                'right_pmtg_abs_r': right_strength,
                'abs_r_difference': abs(right_strength - left_strength),
                'stronger_abs_relationship': stronger_seed,
            })

    selected_pmtg_comparison_df = pd.DataFrame(selected_pmtg_comparison_rows)
    display(selected_pmtg_comparison_df.round({
        'left_pmtg_abs_r': 4,
        'right_pmtg_abs_r': 4,
        'abs_r_difference': 4,
    }))

    y = np.arange(len(target_order))
    bar_height = 0.34
    colors = {'Left pMTG': '#4C78A8', 'Right pMTG': '#F58518'}
    max_abs_r = selected_fc_factor_df['r'].abs().max(skipna=True)
    max_abs_r = 0 if pd.isna(max_abs_r) else max_abs_r
    x_limit = max(0.12, np.ceil((max_abs_r + 0.03) * 100) / 100)

    fig, axes = plt.subplots(
        1,
        len(factor_order),
        figsize=(6.4 * len(factor_order), 5.8),
        sharex=True,
        sharey=True,
    )
    axes = np.atleast_1d(axes)

    for ax, factor_label in zip(axes, factor_order):
        factor_df = selected_fc_factor_df[selected_fc_factor_df['factor_label'].eq(factor_label)]
        r_table = factor_df.pivot(index='target_network', columns='pmtg_seed', values='relationship_strength').reindex(target_order)
        sig_table = factor_df.pivot(index='target_network', columns='pmtg_seed', values='sig_fdr').reindex(target_order)

        for pmtg_label, offset in zip(pmtg_order, [-bar_height / 2, bar_height / 2]):
            values = r_table[pmtg_label].astype(float)
            ax.barh(
                y + offset,
                values,
                height=bar_height,
                color=colors[pmtg_label],
                label=pmtg_label,
                alpha=0.95,
            )
            for y_pos, value, is_sig in zip(y + offset, values, sig_table[pmtg_label]):
                if pd.isna(value):
                    continue
                x_text = value + 0.004
                ha = 'left'
                sig_marker = '*' if bool(is_sig) else ''
                ax.text(
                    x_text,
                    y_pos,
                    f'{value:.3f}{sig_marker}',
                    ha=ha,
                    va='center',
                    fontsize=9,
                    clip_on=False,
                )

        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlim(0, x_limit)
        ax.set_title(factor_label, fontsize=14)
        ax.set_xlabel('Absolute Pearson r With EFA Factor 1', fontsize=11)
        ax.grid(axis='x', alpha=0.25)
        ax.grid(False, axis='y')

    axes[0].set_yticks(y)
    axes[0].set_yticklabels(target_order, fontsize=10)
    axes[0].invert_yaxis()
    axes[0].set_ylabel('Target Network', fontsize=11)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.98))
    fig.suptitle('EFA Factor 1: Left Versus Right pMTG FC To Selected Networks', fontsize=16, y=1.04)
    fig.text(
        0.5,
        0.01,
        'Asterisks Mark FDR-Significant Correlations',
        ha='center',
        fontsize=10,
    )
    plt.tight_layout(rect=[0, 0.04, 1, 0.93])
    plt.show()


In [ ]:
# print the correlation between the two cognitive EFA factors
if len(cognitive_efa_measures) >= 2:
    efa_factor_corr = pg.corr(df[cognitive_efa_measures[0]], df[cognitive_efa_measures[1]])
    print(f"Correlation between {cognitive_efa_measures[0]} and {cognitive_efa_measures[1]}:")
    print(efa_factor_corr)


# Subtype Sizes

In [ ]:
# Count subtype sizes for each clustering solution used in the paper figures.
kmeans_group_counts = df['kmeans_2_consensus'].value_counts()
print(kmeans_group_counts)

louvain_group_counts = df['louvain_consensus'].value_counts()
print(louvain_group_counts)

infomap_group_counts = df['infomap_community_2'].value_counts()
print(infomap_group_counts)


# Mediation Models and SES

In [ ]:
# Test mediation models only when INR is an exposure rather than an already-controlled covariate.
print('Mediation Models Testing INR -> FC -> Cognitive Measures')

if RESIDUALIZE_FOR_INR:
    print('Skipping mediation because this notebook residualizes brain-behavior variables for INR.')
    summary_df = pd.DataFrame()
elif INR_VARIABLE not in df.columns:
    print('Skipping mediation because INR is not available in this no-INR-residualization dataframe.')
    summary_df = pd.DataFrame()
else:
    df = df.loc[:, ~df.columns.duplicated()].copy()
    sig_pairs = cognitive_efa_results_df.loc[
        cognitive_efa_results_df['sig_bonf'], ['col', 'measure']
    ].drop_duplicates()
    summary_results = []

    for _, row in sig_pairs.iterrows():
        mediator = row['col']
        task = row['measure']

        if len({'inr', mediator, task}) < 3:
            continue

        model_df = pd.DataFrame({'inr': df['inr'], mediator: df[mediator], task: df[task]}).dropna().copy()
        if len(model_df) < 5:
            print(f'Skipping mediation for {task} via {mediator}: only {len(model_df)} complete rows')
            continue

        med = pg.mediation_analysis(data=model_df, x='inr', m=mediator, y=task, alpha=0.05, n_boot=5000)

        a_row = med.iloc[0]
        b_row = med.iloc[1]
        total_row = med.loc[med['path'].eq('Total')].iloc[0]
        direct_row = med.loc[med['path'].eq('Direct')].iloc[0]
        indirect_row = med.loc[med['path'].str.contains('Indirect', case=False, na=False)].iloc[0]
        percent_mediated = (indirect_row['coef'] / total_row['coef']) * 100 if total_row['coef'] != 0 else np.nan

        print()
        print(f'Task: {task}')
        print(f'Mediator: {mediator}')
        print(med)

        if indirect_row['sig'] == 'Yes':
            print('Significant mediation')
            print(f'Percent mediated: {percent_mediated:.2f}%')
            summary_results.append({
                'x': 'inr',
                'task': task,
                'mediator': mediator,
                'n': len(model_df),
                'a': a_row['coef'],
                'b': b_row['coef'],
                'direct_effect': direct_row['coef'],
                'indirect_effect': indirect_row['coef'],
                'total_effect': total_row['coef'],
                'percent_mediated': percent_mediated,
                'indirect_pval': indirect_row['pval'],
                'indirect_ci_lower': indirect_row['CI[2.5%]'],
                'indirect_ci_upper': indirect_row['CI[97.5%]'],
            })

    summary_df = pd.DataFrame(summary_results)
    print()
    print('Summary of Significant Mediation Results:')
    if summary_df.empty:
        print('None')
    else:
        print(summary_df.sort_values('indirect_pval'))


# Comparing Clustering Algorithms

In [ ]:
# Compare agreement among Louvain, Infomap, and k-means subtype assignments.
subtype_cols = ['louvain_consensus', 'infomap_community_2', 'kmeans_2_consensus']
matrix = np.zeros((len(subtype_cols), len(subtype_cols)))

for i in range(len(subtype_cols)):
    for j in range(len(subtype_cols)):
        matrix[i, j] = rand_score(df[subtype_cols[i]], df[subtype_cols[j]])

labels = ['Louvain', 'Infomap', 'K-Means']
sns.heatmap(matrix, annot=True, cmap='Blues', xticklabels=labels, yticklabels=labels, fmt='.3f', cbar=False, annot_kws={'size': 16, 'font': 'Arial'})

# Summarize stability across Louvain permutation columns, if they are present.
louvain_perm_cols = [col for col in df.columns if 'louvain_community' in col and 'consensus' not in col and 'subtypes' not in col]
print(len(louvain_perm_cols))
louvain_rand_values = pairwise_rand_scores(df, louvain_perm_cols) if len(louvain_perm_cols) > 1 else []
if louvain_rand_values:
    print(f'Mean Rand Index between louvain permutation columns: {np.mean(louvain_rand_values):.44f}')
else:
    print('Not enough Louvain permutation columns to calculate pairwise Rand scores.')

# Summarize stability across k-means runs, if they are present.
kmeans_perm_cols = [col for col in df.columns if 'kmeans' in col and '_run_' in col and 'consensus' not in col and 'subtypes' not in col]
print(len(kmeans_perm_cols))
kmeans_rand_values = pairwise_rand_scores(df, kmeans_perm_cols) if len(kmeans_perm_cols) > 1 else []
if kmeans_rand_values:
    print(f'Mean Rand Index between kmeans permutation columns: {np.mean(kmeans_rand_values):.44f}')
else:
    print('Not enough k-means run columns to calculate pairwise Rand scores.')

if louvain_rand_values:
    print('Louvain Permutation Rand Scores:')
    print('Range:', min(louvain_rand_values), '-', max(louvain_rand_values))
    print('Mean:', np.mean(louvain_rand_values))
    print('SD:', np.std(louvain_rand_values))

if kmeans_rand_values:
    print('KMeans Run Rand Scores:')
    print('Range:', min(kmeans_rand_values), '-', max(kmeans_rand_values))
    print('Mean:', np.mean(kmeans_rand_values))
    print('SD:', np.std(kmeans_rand_values))
